In [1]:
import os

os.environ["POLARS_MAX_THREADS"] = "64"
import polars as pl

print("Polars threads:", pl.thread_pool_size())

import pandas as pd
import numpy as np

Polars threads: 64


In [20]:
nmf_patterns_non_zero_weights = (
    pl.scan_parquet(
        "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/batched_nmf/all_patterns_partitioned"
    )
    .filter(pl.col("loading") > 0)
    .with_columns([pl.struct(["run_id", "pattern"]).hash().alias("pattern_uid")])
    .with_columns(
        pl.col("pattern_uid").rank(method="dense").cast(pl.UInt32).alias("row_idx") - 1
    )  # needs to be 0-indexed
    .drop("pattern_uid") # don't need this anymore, row_idx is unique identifier
    .collect()
    .rename({"loading": "weight", "pattern": "pattern_num"})
)
nmf_patterns_non_zero_weights

run_id,pattern_num,gene,weight,seed,tol,L1,k,row_idx
i32,i32,str,f64,i32,f64,f64,i64,u32
1,1,"""FBXL18""",0.000058,42,0.00001,0.1,10,33670
1,2,"""FBXL18""",0.000039,42,0.00001,0.1,10,33366
1,3,"""FBXL18""",0.000075,42,0.00001,0.1,10,41064
1,4,"""FBXL18""",0.000029,42,0.00001,0.1,10,3570
1,5,"""FBXL18""",0.000063,42,0.00001,0.1,10,21630
…,…,…,…,…,…,…,…,…
749,53,"""DHRSX""",0.000061,2025,0.0000001,0.7,90,31596
749,59,"""DHRSX""",0.000046,2025,0.0000001,0.7,90,19269
749,67,"""DHRSX""",0.000276,2025,0.0000001,0.7,90,23669


In [ ]:
snrna_nonzero_expression = pl.scan_parquet(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/snrna_nonzero_counts_tall.parquet"
).collect()
snrna_nonzero_expression

cell_id,gene_id,rawcount,logcount
cat,cat,u32,f64
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000188976""",1,0.670818
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000188290""",2,1.126941
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000131591""",1,0.670818
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000078808""",1,0.670818
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000131584""",2,1.126941
…,…,…,…
"""19_TTTGTTGTCTTAAGGC-1""","""ENSG00000198804""",5,1.194889
"""19_TTTGTTGTCTTAAGGC-1""","""ENSG00000198712""",1,0.330965
"""19_TTTGTTGTCTTAAGGC-1""","""ENSG00000198938""",2,0.599993


In [5]:
gene_meta = pl.read_parquet("/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/gene_meta.parquet")
gene_meta

source,type,gene_id,gene_version,gene_name,gene_type,binomial_deviance
cat,cat,cat,cat,cat,cat,f64
"""HAVANA""","""gene""","""ENSG00000243485""","""5""","""MIR1302-2HG""","""lncRNA""",null
"""HAVANA""","""gene""","""ENSG00000237613""","""2""","""FAM138A""","""lncRNA""",null
"""HAVANA""","""gene""","""ENSG00000186092""","""6""","""OR4F5""","""protein_coding""",null
"""HAVANA""","""gene""","""ENSG00000238009""","""6""","""AL627309.1""","""lncRNA""",8778.743973
"""HAVANA""","""gene""","""ENSG00000239945""","""1""","""AL627309.3""","""lncRNA""",914.70276
…,…,…,…,…,…,…
"""ENSEMBL""","""gene""","""ENSG00000277836""","""1""","""AC141272.1""","""protein_coding""",null
"""ENSEMBL""","""gene""","""ENSG00000278633""","""1""","""AC023491.2""","""protein_coding""",null
"""ENSEMBL""","""gene""","""ENSG00000276017""","""1""","""AC007325.1""","""protein_coding""",null


In [31]:
# For each gene, how many patterns (or rows) have weight > 0
nmf_gene_pct = nmf_patterns_non_zero_weights.group_by("gene").agg(
    [pl.count("row_idx").alias("nonzero_patterns")]
)

total_patterns = nmf_patterns_non_zero_weights.select(
    pl.col("row_idx").n_unique()
).item()

nmf_gene_pct = (
    nmf_gene_pct.with_columns(
        [
            (pl.col("nonzero_patterns") / total_patterns * 100).alias(
                "pct_nonzero_weight"
            )
        ]
    )
    .rename({"gene": "gene_name"})
    .with_columns(pl.col("gene_name").cast(pl.Categorical))
)

nmf_gene_pct

gene_name,nonzero_patterns,pct_nonzero_weight
cat,u32,f64
"""AC009005.1""",2707,6.562424
"""CNGA4""",3390,8.218182
"""OR5H14""",2439,5.912727
"""LINC01545""",3092,7.495758
"""AC099654.15""",1502,3.641212
…,…,…
"""DUSP18""",6902,16.732121
"""ARHGAP44""",20696,50.172121
"""TMED9""",8560,20.751515


In [29]:
total_cells = snrna_nonzero_expression.select(
    pl.col("cell_id").n_unique()
).item()

snrna_gene_pct = (
    snrna_nonzero_expression
    .group_by("gene_id")
    .agg([
        pl.count("cell_id").alias("nonzero_cells")
    ])
    .with_columns([
        (pl.col("nonzero_cells") / total_cells * 100).alias("pct_nonzero_expr")
    ])
)
snrna_gene_pct = snrna_gene_pct.join(
    gene_meta.select(["gene_id", "gene_name"]),
    on="gene_id",
    how="left"
)


snrna_gene_pct


gene_id,nonzero_cells,pct_nonzero_expr,gene_name
cat,u32,f64,cat
"""ENSG00000096433""",1622,2.090098,"""ITPR3"""
"""ENSG00000261602""",149,0.192,"""AC092115.2"""
"""ENSG00000204311""",7938,10.228854,"""PJVK"""
"""ENSG00000197859""",6195,7.982836,"""ADAMTSL2"""
"""ENSG00000225676""",53,0.068295,"""AC002378.1"""
…,…,…,…
"""ENSG00000286811""",2517,3.24339,"""AL353138.1"""
"""ENSG00000121057""",18398,23.707541,"""AKAP1"""
"""ENSG00000261798""",25,0.032215,"""AL033527.3"""


In [32]:
df_plot = nmf_gene_pct.join(
    snrna_gene_pct,
    on="gene_name",
    how="inner"
)

df_plot

gene_name,nonzero_patterns,pct_nonzero_weight,gene_id,nonzero_cells,pct_nonzero_expr
cat,u32,f64,cat,u32,f64
"""ITPR3""",2888,7.001212,"""ENSG00000096433""",1622,2.090098
"""AC092115.2""",2746,6.65697,"""ENSG00000261602""",149,0.192
"""PJVK""",6343,15.37697,"""ENSG00000204311""",7938,10.228854
"""ADAMTSL2""",5419,13.13697,"""ENSG00000197859""",6195,7.982836
"""AC002378.1""",2225,5.393939,"""ENSG00000225676""",53,0.068295
…,…,…,…,…,…
"""AL353138.1""",2929,7.100606,"""ENSG00000286811""",2517,3.24339
"""AKAP1""",14003,33.946667,"""ENSG00000121057""",18398,23.707541
"""AL033527.3""",1312,3.180606,"""ENSG00000261798""",25,0.032215


In [37]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool

output_notebook()

source = ColumnDataSource(df_plot.to_pandas())

p = figure(
    title="Percent Non-Zero NMF Weight vs Percent Non-Zero Expression",
    x_axis_label="Percent non-zero expression",
    y_axis_label="Percent non-zero NMF weight",
    tools="pan,wheel_zoom,box_zoom,reset,hover,save"
)

p.scatter(
    x="pct_nonzero_expr",
    y="pct_nonzero_weight",
    size=6,
    source=source,
    alpha=0.5
)

hover = p.select_one(HoverTool)
hover.tooltips = [
    ("Gene", "@gene_name"),
    ("Pct expr", "@pct_nonzero_expr{0.0}"),
    ("Pct NMF weight", "@pct_nonzero_weight{0.0}")
]

show(p)


Loading BokehJS ...